In [1]:
!pip install pyspark -q
from google.colab import drive
drive.mount('/content/drive')

import os, datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Cấu hình đường dẫn chuẩn của Leader
BASE_PATH = "/content/drive/MyDrive/HM-DATA/"
INPUT_FILE = BASE_PATH + "processed/cleaned_transactions.parquet"
OUTPUT_DIR = BASE_PATH + "outputs/candidates/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Khởi tạo Spark
spark = SparkSession.builder \
    .appName("HM_Purchase_History_Pipeline") \
    .config("spark.driver.memory", "10g") \
    .getOrCreate()

print("✅ Spark Ready! Đang trích xuất lịch sử mua sắm...")

Mounted at /content/drive
✅ Spark Ready! Đang trích xuất lịch sử mua sắm...


In [2]:
# 1. Đọc dữ liệu
transactions = spark.read.parquet(INPUT_FILE)

# 2. Xác định mốc thời gian (Chỉ lấy lịch sử TRƯỚC tuần Validation)
max_date = transactions.select(F.max("t_dat")).collect()[0][0]
val_start_date = max_date - datetime.timedelta(days=14)

# 3. Trích xuất lịch sử độc nhất, sắp xếp theo thời gian mới nhất
# Chúng ta dùng Window để lấy những món mua gần đây nhất lên đầu
window_spec = Window.partitionBy("customer_id").orderBy(F.desc("t_dat"))

history_candidates = transactions.filter(F.col("t_dat") < F.lit(val_start_date)) \
    .withColumn("article_id", F.lpad(F.col("article_id").cast("string"), 10, "0")) \
    .withColumn("rn", F.row_number().over(window_spec)) \
    .filter(F.col("rn") <= 20) # Lấy 20 món gần nhất

# 4. Gom lại thành mảng (Array) cho mỗi Customer
history_candidates_df = history_candidates.groupBy("customer_id") \
    .agg(F.collect_list("article_id").alias("history_candidates"))

# 5. Lưu file Parquet
history_candidates_df.write.mode("overwrite").parquet(OUTPUT_DIR + "history_candidates_W7.parquet")

print(f"✅ Đã tạo xong ứng viên Lịch sử cho {history_candidates_df.count():,} khách hàng.")

✅ Đã tạo xong ứng viên Lịch sử cho 1,350,492 khách hàng.


In [3]:
# 1. Ground Truth (Tuần 7)
ground_truth_w7 = transactions.filter(
    (F.col("t_dat") >= F.lit(val_start_date)) &
    (F.col("t_dat") < F.lit(max_date - datetime.timedelta(days=7)))
).select("customer_id", F.lpad(F.col("article_id").cast("string"), 10, "0").alias("article_id"))

actual_counts = ground_truth_w7.groupBy("customer_id").count().withColumnRenamed("count", "actual_cnt")

# 2. Explode ứng viên lịch sử
candidates_exploded = history_candidates_df.select(
    "customer_id",
    F.explode("history_candidates").alias("article_id")
)

# 3. Tính Hits và Recall
hits = ground_truth_w7.join(candidates_exploded, ["customer_id", "article_id"], "inner") \
    .groupBy("customer_id").count().withColumnRenamed("count", "hit_cnt")

recall_stats = actual_counts.join(hits, "customer_id", "left").fillna(0)
final_recall_history = recall_stats.select(F.avg(F.col("hit_cnt") / F.col("actual_cnt"))).collect()[0][0]

print("-" * 50)
print(f"📊 KẾT QUẢ RECALL NHÁNH PURCHASE HISTORY (N=20)")
print(f"Average Recall@20: {final_recall_history:.6f}")
print("-" * 50)

--------------------------------------------------
📊 KẾT QUẢ RECALL NHÁNH PURCHASE HISTORY (N=20)
Average Recall@20: 0.056600
--------------------------------------------------
